# Baseline and Correction Preview

Inspect how the saved ML correction model changes picks on one held-out segment.

By default this notebook picks a test segment with near-median MAE improvement from `reports/evaluation/test_segment_metrics.csv`.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display
import matplotlib.pyplot as plt, numpy as np, pandas as pd

This first setup cell loads the general-purpose tools for tables, plots, and notebook display. It is intentionally lightweight: enough to inspect one segment carefully without dragging in the full training pipeline. By the end of it, the notebook has the basic pieces needed to compare baseline and corrected picks.

In [ ]:
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "seismic_first_break_picker").exists():
    repo_root = repo_root.parent
if not (repo_root / "seismic_first_break_picker").exists():
    raise RuntimeError("Could not locate the repository root. Start Jupyter from anywhere inside this repo.")
for path in (repo_root, repo_root / "notebooks"):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)



from _notebook_setup import INTERIM_DIR, PROCESSED_DIR, REPORTS_DIR, require_path
from seismic_first_break_picker.correction import extract_patch, predict_corrected_panel
from seismic_first_break_picker.data import load_segment
from seismic_first_break_picker.metrics import compute_metric_bundle
from seismic_first_break_picker.modeling import load_model_artifact

%matplotlib inline
plt.style.use("default")

This cell does the heavy lifting for reproducibility. It locates the repository root, wires up import paths, and loads the exact project functions that the shipped pipeline uses for correction, patch extraction, metric computation, and model loading. That matters because the notebook is not re-implementing the pipeline; it is interrogating the real one.

## Representative Test Case

With imports and repository paths ready, the next cell chooses a held-out segment from the evaluation CSV. It targets a segment whose MAE improvement is close to the median so the notebook focuses on a typical, not cherry-picked, example.

In [ ]:
segments_dir, model_path, metrics_path = require_path(INTERIM_DIR / "Halfmile_segments"), require_path(PROCESSED_DIR / "ml_correction_model.pkl"), require_path(REPORTS_DIR / "evaluation" / "test_segment_metrics.csv")

metrics_df = pd.read_csv(metrics_path)
median_gain = metrics_df["mae_improvement_ms"].median()
target_row = metrics_df.iloc[(metrics_df["mae_improvement_ms"] - median_gain).abs().argmin()]

display(target_row.to_frame().T[["segment_id", "baseline_mae_ms", "corrected_mae_ms", "mae_improvement_ms"]].round(3))

Here we choose a test case in a defensible way instead of hand-picking a flattering panel. Using the median MAE improvement from the evaluation CSV gives us a segment that is broadly representative of the model's typical impact. The displayed row is a compact summary of what we are about to inspect in more detail.

## Run Baseline And Correction

This cell loads the saved model artifact, opens the selected segment file, and runs the end-to-end correction path. The metric table below shows how much the corrected pick line changes the baseline on this exact segment.

In [ ]:
artifact = load_model_artifact(model_path)
segment_id = target_row["segment_id"]
segment = load_segment(segments_dir / f"{segment_id}.npz")
baseline_idx, corrected_idx = predict_corrected_panel(segment["panel"], artifact)

metric_table = pd.DataFrame(
    [
        {"method": "refined_baseline", **compute_metric_bundle(baseline_idx, segment["fb_idx"], segment["valid"], segment["sample_ms"])},
        {"method": "ml_corrected", **compute_metric_bundle(corrected_idx, segment["fb_idx"], segment["valid"], segment["sample_ms"])},
    ]
)

print(segment["segment_id"])
display(metric_table.round(3))
artifact["selected_config"]

This is the core analytical step of the notebook. We load the persisted model artifact, reopen the original segment, run the refined baseline and the learned correction, and then score both against the manual labels. The result is a segment-level comparison that connects the saved model directly to an interpretable test example.

## Full-Panel Overlay

The next figure puts the manual, baseline, and ML-corrected lines on the same panel. Its purpose is qualitative: to see where the correction stays close to the manual line and where residual bias remains.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.imshow(segment["panel"], cmap="gray", aspect="auto", origin="upper")
valid_trace_ids = np.flatnonzero(segment["valid"])
ax.plot(valid_trace_ids, segment["fb_idx"][segment["valid"]], color="tab:red", linewidth=1.2, label="Manual")
ax.plot(np.arange(len(baseline_idx)), baseline_idx, color="cyan", linewidth=1.0, label="Baseline")
ax.plot(np.arange(len(corrected_idx)), corrected_idx, color="lime", linewidth=1.0, label="Corrected")
ax.set_title(segment["segment_id"])
ax.set_xlabel("Trace index")
ax.set_ylabel("Time sample")
ax.legend()
plt.show()

Once the metrics are on the table, this plot shows the shape of the problem. It lets us see whether the corrected line removes a local bias, whether it overreacts, and where the baseline was already doing fine. Numbers alone can tell us that an improvement happened; this view shows what that improvement actually looks like.

## Local Error And Patch View

This final diagnostic zooms in on the trace with the largest baseline error among valid labels. It plots per-trace absolute error and shows the exact image patch the correction model sees around that trace.

In [ ]:
valid_trace_ids = np.flatnonzero(segment["valid"])
baseline_error_ms = np.abs(baseline_idx[segment["valid"]] - segment["fb_idx"][segment["valid"]]) * segment["sample_ms"]
corrected_error_ms = np.abs(corrected_idx[segment["valid"]] - segment["fb_idx"][segment["valid"]]) * segment["sample_ms"]
worst_trace = int(valid_trace_ids[np.argmax(baseline_error_ms)])

feature_spec = artifact["feature_spec"]
patch = extract_patch(
    panel=segment["panel"],
    center_trace=worst_trace,
    center_time=int(baseline_idx[worst_trace]),
    half_width=int(feature_spec["half_width"]),
    half_height=int(feature_spec["half_height"]),
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)

axes[0].plot(valid_trace_ids, baseline_error_ms, label="Baseline abs error (ms)")
axes[0].plot(valid_trace_ids, corrected_error_ms, label="Corrected abs error (ms)")
axes[0].axvline(worst_trace, color="black", linestyle="--", linewidth=1)
axes[0].set_title("Per-trace error on valid labels")
axes[0].set_xlabel("Trace index")
axes[0].set_ylabel("Absolute error (ms)")
axes[0].legend()

axes[1].imshow(patch, cmap="gray", aspect="auto", origin="upper")
axes[1].set_title(f"Model patch around trace {worst_trace}")
axes[1].set_xlabel("Local trace window")
axes[1].set_ylabel("Local time window")

plt.show()

This final diagnostic moves from full-panel behavior down to the local evidence the model sees. By finding the trace with the largest baseline error and extracting its patch, we connect the model input to the observed correction outcome. That makes the notebook useful not just for demonstration, but for debugging whether the local-patch design is helping in the places we care about.